# 58장 · Progressive Disclosure와 Skill 로더

Progressive Disclosure는 먼저 짧은 목록을 읽고 필요한 지침과 참고자료만 여는 방식입니다. 어떤 파일을 실제로 읽었는지 기록으로 확인합니다.

대출 규정을 답하는 Skill 본문을 사전에서 고릅니다. 제공된 파일 읽기 기록의 바이트 수와 이 수업에서 직접 계산한 문자열 길이를 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 목록 메타데이터와 본문 지연 로딩

처음에는 작업의 이름과 설명만 보여 줍니다. 질문을 받은 뒤 필요한 Skill 본문을 읽고 그 안에서 지정한 참고자료를 엽니다. 전체 파일을 미리 넣는 방식과 같은 질문으로 비교하세요.


### 셀 01 · 이름으로 Skill 본문 조회

chosen에 지정한 이름과 /SKILL.md를 합쳐 사전에서 본문 한 건을 꺼냅니다.


In [1]:
files = {"evidence-answer/SKILL.md": "답변 전 references/source-check.md에서 원문 대조 기준을 읽는다.",
         "comparison-report/SKILL.md": "동일 조건의 두 실행 기록을 비교한다."}
chosen = "evidence-answer"
print(files[f"{chosen}/SKILL.md"])

답변 전 references/source-check.md에서 원문 대조 기준을 읽는다.


대출 질문을 처리할 때 읽은 본문에는 원문 대조 안내가 나옵니다. 비교 보고서의 본문은 이번 입력에 들어가지 않았습니다.

Skill 본문에 참고 파일의 경로가 적혀 있어도 그 파일 내용까지 모델에 전달된 것은 아닙니다. 이 단계에서 읽은 문자열과 후속 호출에서 추가할 문자열을 구분하면, 어떤 지침을 실제로 제공했는지 확인할 수 있습니다.


### 이름으로 본문 하나 열기

<p>57장에서 작성한 지침 중 chosen이 가리키는 파일을 엽니다. 이 셀에서 읽은 본문은 한 개입니다.</p>

<h3>Skill 이름, 선택한 본문, 읽기 기록</h3>

<p>chosen 값 evidence-answer로 files에서 지정한 본문을 찾습니다. 본문에는 references/source-check.md를 읽으라는 지시가 있습니다. 그 파일을 실제로 읽었는지 확인하려면 별도 자료인 reads에서 그 경로를 찾으세요. 본문에 경로가 등장하는 것만으로는 읽었다고 판단할 수 없습니다.</p>


### 셀 02 · 읽기 기록의 바이트 수 집계

앞선 실행 기록의 bytes 필드에서 가져온 두 값을 합산합니다. 이 셀은 파일을 열지 않습니다.


In [2]:
reads = {"evidence-answer/SKILL.md": 2511,
         "evidence-answer/references/source-check.md": 1049}
print(sum(reads.values()))

3560


두 파일의 크기를 더한 3560바이트입니다. 이 값은 모델의 입력 토큰이나 청구 금액이 아닙니다. 실제 파일 읽기 기록은 loading-trace.json에 있습니다.


#### 목록만 읽을 때와 본문을 열 때

<mark>선택에 필요한 설명과 선택 뒤에 필요한 긴 지침은 읽는 시점이 다릅니다.</mark> 요청과 맞는 Skill을 골랐다면 그제야 본문이 가리키는 참조 파일을 엽니다. 이름은 맞지만 파일이 없을 때 어느 단계에서 실패하는지도 구별할 수 있습니다.

선수 개념 · 57장 SKILL.md의 설명·입력·작업 지침

### 읽기 기록과 예제 문자열의 단위

<p>두 파일의 기록을 더하면 2,511 + 1,049 = 3,560바이트입니다. 셀 05의 48과 70은 이 페이지에 실린 축약 문자열의 글자 수입니다. 같은 파일에 대한 두 측정값으로 비교할 수 없습니다.</p>


## Skill 선택 오류와 참조 파일 접근

‘비교’라는 단어 하나만 보고 Skill을 고르면 질문과 맞지 않을 수 있습니다. 선택한 이름, 읽은 참고자료, 인용한 원문 순서로 점검합니다. 선택과 읽기가 성공했어도 답의 의미는 따로 확인하세요.


### 과제 03 · 참조 파일 접근 기록

필수 참고자료가 현재 읽기 기록에 들어 있는지 검사하세요. 본문을 열었다는 사실만으로는 충분하지 않습니다.

True는 제공된 읽기 기록에서 해당 경로를 찾았다는 뜻입니다. 지금 파일을 열어 확인한 결과와 구별하세요. 직접 읽기를 측정할 때에는 파일을 열어 본 뒤 그 시점의 경로를 실행 기록에 남깁니다.


In [3]:
reference = 'evidence-answer/references/source-check.md'
# reads에 reference가 있는지 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reference = &#x27;evidence-answer/references/source-check.md&#x27;
print(reference in reads)</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True는 제공 기록에 그 경로가 있다는 뜻입니다. 답변이 대출 기간을 14일로 적었는지는 기록의 경로만 봐서는 알 수 없습니다.</p>
</details>


### 참조 파일을 열었는지 확인하기

<p>Skill 본문에 참조 파일 이름이 적힌 것과 실제 읽기 기록에 그 파일이 남은 것은 다른 관찰입니다. reads에서 경로를 키로 찾아 두 번째 동작까지 일어났는지 확인하세요.</p>


## load_skill()의 선택 결과와 읽은 분량

모든 본문을 전달하면 모델은 비교 작업용 절차까지 읽습니다. 아래에서는 작업 하나의 본문만 반환하는 함수를 만들고, 잘못 선택했을 때 무엇이 보이는지 확인합니다.


### 셀 04 · 잘못된 Skill 선택 진단

대출 기간 질문에 비교 보고서용 이름을 전달합니다.


In [4]:
def load_skill(name):
    return files[f"{name}/SKILL.md"]

print(load_skill("comparison-report"))

동일 조건의 두 실행 기록을 비교한다.


사전 조회는 성공했지만 comparison-report의 본문에는 대출 규정 확인 절차가 없습니다. 경로 오류에서 생기는 KeyError와 다른 경우입니다.


### load_skill의 인자와 반환값

<p>load_skill("comparison-report")에 넘긴 것은 파일 경로 전체가 아니라 이름입니다. 반환값은 그 이름으로 찾은 본문의 문자열입니다. 이 셀의 files는 파일 읽기를 재현한 딕셔너리입니다.</p>


### 셀 05 · 선택 로딩과 전체 로딩 비교

같은 파일 모음에서 필요한 본문만 뽑았을 때 문자열 길이를 셉니다.


In [5]:
all_instructions = "\n".join(files.values())
selected = load_skill("evidence-answer")
print(len(selected), len(all_instructions))

48 70


첫 수는 대출 답변에 필요한 본문 길이, 둘째 수는 비교 지침까지 붙인 전체 길이입니다. Python의 문자 수이므로 모델의 실제 토큰 사용량은 따로 측정해야 합니다.


#### 이름 하나로 선택한 본문만 전달하기

```mermaid
flowchart TD
nfiles["두 Skill 본문"]
  npick["load_skill"]
  none["선택 본문"]
  nboth["전체 연결"]
  nfiles --> npick
  npick --> none
  nfiles --> nboth
```

같은 files에서 선택한 문자열과 전체 문자열의 길이를 셉니다. len의 단위는 글자입니다.

### 선택한 본문의 분량 비교

<p>마지막 셀은 본문 하나와 두 본문을 줄바꿈으로 합친 문자열의 길이를 비교합니다. 48과 70은 파일 접근 횟수가 아닌 문자열의 글자 수입니다.</p>


## 복습과 실습 점검

### 과제 06 · 선택한 본문의 글자 수와 바이트 수 재기

셀 05에서 선택한 한국어 본문 문자열을 UTF-8로 인코딩하세요. 글자 수와 바이트 수를 같은 줄에 출력해 두 단위가 왜 다른지 확인하세요.


In [ ]:
# selected와 UTF-8로 인코딩한 selected의 길이를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(len(selected), len(selected.encode(&quot;utf-8&quot;)))</code></pre>
<p>실행 결과</p><pre><code>48 78</code></pre>
<p>첫 값은 파이썬 문자열 길이이고 둘째 값은 UTF-8 바이트 수입니다. 한글이 들어 있으면 같은 문자열에서도 두 값이 다릅니다. 모델이 받을 토큰 수를 알려면 사용하는 토크나이저로 같은 문자열을 분할해야 합니다.</p>
</details>


## 요청 한 건에서 실제로 연 Skill 파일 기록하기

서비스에 Skill 두 개가 설치돼 있습니다. 목록을 확인할 때마다 두 본문과 참고자료까지 모두 읽으면 모델에 전달할 내용이 길어집니다. 이번에는 임시 폴더에 실제 파일을 만들고, 자료를 열 때마다 경로를 남깁니다.

### 셀 07 · 선택된 본문만 파일에서 읽기

앞 셀의 files에 있는 두 Skill을 임시 폴더에 저장합니다. 같은 폴더에 원문 검산 지침도 한 개 둡니다. 먼저 단순 비교 문의를 처리한다고 생각하고 어떤 파일이 열렸는지 확인하세요.

In [6]:
from pathlib import Path
from tempfile import TemporaryDirectory

skill_folder = TemporaryDirectory()
root = Path(skill_folder.name)
reference_path = "evidence-answer/references/source-check.md"
example_files = {**files, reference_path: "인용한 문장과 원문을 대조하고 쪽수를 적습니다."}
for relative, content in example_files.items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

def read_step(relative, opened):
    content = (root / relative).read_text(encoding="utf-8")
    opened.append(relative)
    return content

opened = []
print(read_step("comparison-report/SKILL.md", opened))
print(opened)

동일 조건의 두 실행 기록을 비교한다.
['comparison-report/SKILL.md']


파일 읽기가 끝난 뒤 열린 목록에는 comparison-report/SKILL.md 하나만 있습니다. 두 Skill이 같은 폴더에 설치돼 있어도 이 요청에서 사용한 파일은 한 개입니다.

### 과제 08 · 원문 대조가 필요한 요청에서만 참고자료 열기

evidence-answer 본문은 항상 읽고, 원문 대조를 요청한 경우에만 reference_path를 추가로 읽는 함수를 작성하세요. 두 번의 호출에 각각 새 목록을 만들고 원문 대조 전후의 경로를 확인하세요.

In [7]:
# def select_evidence_steps(check_source): ...
# 선택한 본문과 참고자료의 경로를 실제 read_step을 호출해 기록하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def select_evidence_steps(check_source):
    opened = []
    read_step(&quot;evidence-answer/SKILL.md&quot;, opened)
    if check_source:
        read_step(reference_path, opened)
    return opened

print(select_evidence_steps(False))
print(select_evidence_steps(True))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;evidence-answer/SKILL.md&#x27;]
[&#x27;evidence-answer/SKILL.md&#x27;, &#x27;evidence-answer/references/source-check.md&#x27;]</code></pre>
<p>일반 요청에는 본문 1개, 대조 요청에는 본문과 source-check.md 2개가 남습니다. 목록에 경로를 미리 채운 값은 실제 읽기 기록으로 쓸 수 없습니다. 이 예에서 opened는 파일 읽기가 성공한 뒤에만 갱신됩니다.</p>
</details>

<mark>설치된 파일 수와 이번 요청에서 연 파일 수는 다릅니다.</mark> 참고자료가 작아도 필요 없는 요청에서 매번 열면 모든 요청에 그 분량이 더해집니다. 경로 목록은 무엇을 읽었는지 알려 줍니다. 토큰 수는 별도로 측정해야 합니다.

서비스 로그에 선택한 Skill과 실제로 읽은 파일 경로를 따로 남길 수 있습니다.